# CymbalGoal—Intelligent Search: From Keywords to Hybrid
## SOLUTION NOTEBOOK

**This is the completed version.** Every query from the lab guide is here, in order, with a
comment at the top of each cell naming the lab section it came from.

Use it to check an answer, to remember what a result looked like, or to read the whole flow
end to end. Working through the lab yourself first is worth considerably more—the interesting
parts of Tasks 2 through 5 are the failures, and they land harder when you run into them
rather than read about them.

Tasks 2 through 5 are cells you would normally add yourself. They are pre-written here.


## Task 1.1—Connect to your cluster

### Install the client libraries

Two packages do the work:

- **`google-cloud-alloydb-connector`**—Google's AlloyDB connector. It handles the TLS handshake and
  the IAM token exchange, so you never build a connection string or hold a password.
- **`pg8000`**—a pure-Python PostgreSQL driver. The connector hands it an already-authenticated,
  already-encrypted socket.

This takes about thirty seconds, and it is the longest install in the lab.

In [ ]:
# Lab Task 1.1, step 1 — install the AlloyDB connector and pandas

!pip install -q "google-cloud-alloydb-connector[pg8000]" pandas 2>&1 | tail -1
print("client libraries ready")

### Discover your environment

Notice there is nothing to fill in below. No project ID, no region, no cluster name, no username.

Everything is discovered from the environment this runtime is already running in. That is what lets
the same notebook work for you, for the person beside you, and for you again in your own project
months from now—and it is why this file can live in a public GitHub repository without anyone
needing a credentials review.

**Read the code before you run it.** It is worth seeing how little is actually required.

In [ ]:
# Lab Task 1.1, step 2 — discover project, region, cluster, instance and identity

import subprocess, json, time, io, gzip, csv, textwrap

def sh(cmd):
    """Run a shell command and return its trimmed stdout."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

PROJECT = sh("gcloud config get-value project")
USER    = sh("gcloud config get-value account")

# Find the cluster rather than assuming its name. A hardcoded name is a notebook
# that works exactly once, in exactly one project.
clusters = json.loads(sh("gcloud alloydb clusters list --format=json") or "[]")
assert clusters, "No AlloyDB cluster found in this project. Is the lab still provisioning?"
cluster  = clusters[0]
CLUSTER  = cluster["name"].split("/")[-1]
REGION   = cluster["name"].split("/locations/")[1].split("/")[0]

instances = json.loads(
    sh(f"gcloud alloydb instances list --cluster={CLUSTER} --region={REGION} --format=json") or "[]")
primary = [i for i in instances if i.get("instanceType") == "PRIMARY"]
assert primary, f"No PRIMARY instance found in cluster {CLUSTER}."
INSTANCE = primary[0]["name"].split("/")[-1]

INSTANCE_URI = f"projects/{PROJECT}/locations/{REGION}/clusters/{CLUSTER}/instances/{INSTANCE}"
DB_NAME = "cymbalgoal"
GCS     = "gs://class-demo/alloydb-labs/cymbalgoal"

print(f"project   {PROJECT}")
print(f"region    {REGION}")
print(f"cluster   {CLUSTER}")
print(f"instance  {INSTANCE}")
print(f"you       {USER}")
print(f"\ntarget    {INSTANCE_URI}")
print("\nNote what is absent from that output: a password.")

### Connect, and build the helpers you will use all lab

`enable_iam_auth=True` is the whole trick. Instead of sending a username and password, the connector
exchanges your Google Cloud credentials for a short-lived token. AlloyDB verifies that token and looks
you up in its own list of IAM users. Your Google identity *is* your database login.

Three helpers get defined here. The one you will use most is **`q()`**:

| Helper | What it does |
| :---- | :---- |
| `q(sql)` | Run a query, return the results as a pandas DataFrame |
| `run(sql)` | Run a statement that returns no rows—DDL, `COPY`, `SET` |
| `explain(sql)` | Show the query plan. You will need this in Tasks 3 and 4 |

**`q()` is what Tasks 2 through 5 are built on.** When the lab hands you a query, add a new cell and
run it through `q()`.

In [ ]:
# Lab Task 1.1, step 3 — connect with IAM auth and define q() / run() / explain()

from google.cloud.alloydb.connector import Connector, IPTypes
import pandas as pd

connector = Connector()
_conn = None   # one session, reused across cells, reconnected if it drops

def _connect(db=DB_NAME):
    c = connector.connect(
        INSTANCE_URI, "pg8000",
        user=USER, db=db,
        enable_iam_auth=True,          # no password anywhere
        ip_type=IPTypes.PUBLIC,        # IAM gates access; the connector carries mTLS
    )
    c.autocommit = True
    return c

def _session():
    """Return a live connection, reconnecting if the previous one died."""
    global _conn
    if _conn is None:
        _conn = _connect()
        return _conn
    try:
        cur = _conn.cursor(); cur.execute("SELECT 1"); cur.close()
    except Exception:
        _conn = _connect()
    return _conn

def _notices(conn, show):
    # AlloyDB reports index-build statistics as PostgreSQL NOTICEs. pg8000 hands
    # them over as raw dicts with BYTE keys -- {b'S': b'NOTICE', b'M': b'...'} --
    # which is unreadable on a projector. Decode the message field.
    #
    # ALWAYS drain the queue, whether or not we print it. pg8000 accumulates
    # notices on the connection until something clears them, so skipping the
    # drain means the first cell that asks for notices dumps every "already
    # exists, skipping" message from every cell before it.
    queued = getattr(conn, "notices", None) or []
    if show:
        for n in queued:
            msg = n.get(b"M")
            if msg:
                print("   ", msg.decode())
    try:
        queued.clear()
    except Exception:
        pass

def run(sql, db=None, show_notices=False):
    """Execute a statement that returns no rows."""
    conn = _connect(db) if db else _session()
    cur = conn.cursor()
    cur.execute(sql)
    cur.close()
    _notices(conn, show_notices)
    if db:
        conn.close()

def q(sql, show_notices=False):
    """Run a query and return a pandas DataFrame. This is your main tool."""
    conn = _session()
    cur = conn.cursor()
    cur.execute(sql)
    cols = [d[0] for d in cur.description] if cur.description else []
    rows = cur.fetchall() if cur.description else []
    cur.close()
    _notices(conn, show_notices)
    return pd.DataFrame(rows, columns=cols)

def explain(sql, analyze=True):
    """Print the query plan. Use this to prove an index is actually being used."""
    kw = "EXPLAIN (ANALYZE, BUFFERS)" if analyze else "EXPLAIN"
    conn = _session()
    cur = conn.cursor()
    cur.execute(f"{kw} {sql}")
    for row in cur.fetchall():
        print(row[0])
    cur.close()

# Connect to the default 'postgres' database first -- 'cymbalgoal' does not exist yet.
t0 = time.time()
probe = _connect("postgres")
cur = probe.cursor(); cur.execute("SELECT version()"); ver = cur.fetchone()[0]; cur.close()
probe.close()
print(f"connected in {time.time()-t0:.1f}s")
print(ver.split(" on ")[0])

---

⏸ **Stop here.** You have finished the notebook's Task 1.1.

Return to the lab guide for **Task 1.2**, which explains why creating the database is your job and not the provisioning's.

## Task 1.2—Create your database and enable four extensions

### Create your database

Your cluster arrived with only the default `postgres` database. You are about to create your own.

**Why you, and not the provisioning that built the cluster?** Because a database created by
automation is owned by `postgres`. AlloyDB's `alloydbsuperuser` role is deliberately *not* a true
PostgreSQL superuser, so you would not be able to drop or fully manage a database you supposedly
owned. Creating it yourself makes you the owner, with everything that implies.

The cell below is **guarded**: it checks whether the database already exists before creating it, and
it never drops anything. Re-running cells is normal, and no cell in this lab will punish you for it.

In [ ]:
# Lab Task 1.2 — create the cymbalgoal database (guarded; never drops anything)

conn = _connect("postgres")
cur = conn.cursor()
cur.execute("SELECT pg_catalog.pg_get_userbyid(datdba) FROM pg_database WHERE datname = %s", (DB_NAME,))
row = cur.fetchone()

if row is None:
    cur.execute(f'CREATE DATABASE "{DB_NAME}"')
    print(f"created database {DB_NAME}, owned by {USER}")
else:
    owner = row[0]
    if owner == USER:
        print(f"database {DB_NAME} already exists and you own it—nothing to do")
    else:
        raise SystemExit(
            f"database {DB_NAME} exists but is owned by '{owner}', not you ({USER}).\n"
            f"You would not be able to manage it. Have the owner drop it, then re-run this cell."
        )
cur.close(); conn.close()

### Enable the extensions

PostgreSQL ships deliberately small, and capabilities arrive as extensions. Four matter here:

| Extension | What it gives you | First used in |
| :---- | :---- | :---- |
| `vector` | The `VECTOR` column type for storing embeddings | Task 1 |
| `alloydb_scann` | Google's ScaNN index for fast similarity search | Task 1 |
| `google_ml_integration` | Calling models from inside SQL—`ai.embedding()`, `ai.rank()` | Task 2 |
| `pg_textsearch` | BM25 relevance ranking for full-text search | Task 3 |

`pg_textsearch` and `alloydb_scann` both require the `alloydbsuperuser` role. You already hold it—provisioning granted it to your IAM identity, which is why the next cell simply works.

In [ ]:
# Lab Task 1.2 — enable the four extensions this lab needs

for ext in ["vector", "alloydb_scann", "google_ml_integration", "pg_textsearch"]:
    run(f"CREATE EXTENSION IF NOT EXISTS {ext}")

display(q("""
    SELECT extname AS extension, extversion AS version
    FROM pg_extension
    WHERE extname IN ('vector','alloydb_scann','google_ml_integration','pg_textsearch')
    ORDER BY extname
"""))

---

⏸ **Stop here.** You have finished the notebook's Task 1.2.

Return to the lab guide for **Task 1.3**, and the schema you are about to apply.

## Task 1.3—Apply the schema

The schema is staged in Cloud Storage as `schema.sql`: eight tables, their constraints, and
thirty-eight `COMMENT ON` statements describing what each column means.

Those comments are not decoration. Point a language model at a schema like this one and ask it to
write SQL, and comments like these are a large part of what makes its answers correct—they record that
player names are *not* unique, and that a NULL `transfer_fee` means "unknown" while zero means "free."

**Notice what `schema.sql` does not contain: a single `CREATE INDEX`.** That is Task 1.6, and the
reason is worth waiting for.

In [ ]:
# Lab Task 1.3 — fetch the schema from Cloud Storage and read what it contains

schema_sql = sh(f"gcloud storage cat {GCS}/schema.sql")
assert schema_sql.strip(), "could not read schema.sql from Cloud Storage"

print(f"schema.sql is {len(schema_sql):,} characters")
print(f"  CREATE TABLE statements: {schema_sql.upper().count('CREATE TABLE')}")
print(f"  COMMENT ON statements:   {schema_sql.upper().count('COMMENT ON')}")
print(f"  CREATE INDEX statements: {schema_sql.upper().count('CREATE INDEX')}   <- deliberately zero")

# The table this whole lab revolves around, in full.
start = schema_sql.upper().find("CREATE TABLE PLAYERS")
end   = schema_sql.find("\n);", start)
print("\n" + "=" * 78)
print(schema_sql[start:end + 3])

# Three comments that carry real weight. These are not documentation for humans.
print("=" * 78)
print("A few of the 38 column comments:\n")
for line in schema_sql.splitlines():
    s = line.strip()
    if s.upper().startswith("COMMENT ON") and any(
            k in s for k in ("player_name", "transfer_fee", "to_club_id")):
        print(textwrap.fill(s, 78, subsequent_indent="    "), "\n")

In [ ]:
# Lab Task 1.3 — apply the schema (8 tables, 38 comments, 0 indexes)

# schema_sql was fetched and displayed in the cell above.
EXPECTED = ["competitions", "clubs", "players", "games",
            "appearances", "game_events", "player_valuations", "transfers"]

present = q(f"""
    SELECT count(*) AS n FROM information_schema.tables
    WHERE table_schema = 'public' AND table_type = 'BASE TABLE'
      AND table_name IN ({','.join("'" + t + "'" for t in EXPECTED)})
""")["n"][0]

if present == len(EXPECTED):
    print(f"\nall {present} tables already exist—skipping (schema.sql is not re-runnable)")
elif present == 0:
    t0 = time.time()
    run(schema_sql)
    print(f"\nschema applied in {time.time()-t0:.1f}s")
else:
    raise SystemExit(
        f"\n{present} of {len(EXPECTED)} tables exist. That is a half-built schema.\n"
        f"Drop the ones that exist, or drop and recreate the database, then re-run."
    )

display(q("""
    SELECT table_name,
           (SELECT count(*) FROM information_schema.columns c
             WHERE c.table_name = t.table_name AND c.table_schema = 'public') AS columns
    FROM information_schema.tables t
    WHERE table_schema = 'public' AND table_type = 'BASE TABLE'
      AND left(table_name, 1) <> '_'
    ORDER BY table_name
"""))

---

⏸ **Stop here.** You have finished the notebook's Task 1.3.

Return to the lab guide for **Task 1.4**, which explains what this dataset actually is before you load 1.6 million rows of it.

## Task 1.4—Load the data

### Preflight the load

Before moving a single byte, this cell compares the number of fields in each staged file against the
column list we intend to load it into, and refuses to continue on a mismatch.

**This guard is scar tissue, not paranoia.** A column list *longer* than the file fails loudly and
harmlessly. A list of the *same length in a different order* loads silently—putting stadium capacity
into market value—and the first symptom is a query returning nonsense in front of a room. Cheap
check, expensive alternative.

One subtlety it handles: the manifest's `column_order` is derived from the table definition, so for
`players` and `clubs` it includes `profile_text` and `profile_embedding`—columns the relational files do
not carry, because those arrive separately in the profile load below. Those get subtracted here.

In [ ]:
# Lab Task 1.4 — define copy_table(), then preflight every file against its column list

# ---------------------------------------------------------------------------
# copy_table — the bulk loader every step below uses.
#
# COPY is PostgreSQL's bulk path: it hands the server one stream to parse,
# instead of planning and executing a statement per row. On `appearances`
# (832,193 rows) that is the difference between a minute and an afternoon.
#
# The file never lands on this runtime's disk. `gcloud storage cat` streams it,
# gzip decompresses in flight, and pg8000 pushes it straight into COPY FROM STDIN.
# ---------------------------------------------------------------------------
def copy_table(table, cols, gcs_uri):
    """Stream a gzipped CSV from Cloud Storage directly into `table`."""
    # Peek at row one so a header row is detected rather than assumed. A header
    # loaded as data becomes one corrupt row that survives every count check.
    peek = subprocess.Popen(["gcloud", "storage", "cat", gcs_uri], stdout=subprocess.PIPE)
    with gzip.GzipFile(fileobj=peek.stdout, mode="rb") as gz:
        first = next(csv.reader(io.TextIOWrapper(gz, encoding="utf-8")))
    peek.stdout.close(); peek.wait()
    has_header = [c.strip().lower() for c in first] == [c.strip().lower() for c in cols]

    conn = _session()
    cur  = conn.cursor()
    proc = subprocess.Popen(["gcloud", "storage", "cat", gcs_uri], stdout=subprocess.PIPE)
    try:
        with gzip.GzipFile(fileobj=proc.stdout, mode="rb") as gz:
            cur.execute(
                f'COPY {table} ({", ".join(cols)}) FROM STDIN '
                f'WITH (FORMAT csv, HEADER {"true" if has_header else "false"})',
                stream=gz,
            )
        conn.commit()
    finally:
        cur.close()
        proc.stdout.close(); proc.wait()


PASS2 = {"profile_text", "profile_embedding"}
ORDER = ["competitions", "clubs", "players", "games",
         "appearances", "game_events", "player_valuations", "transfers"]

manifest = json.loads(sh(f"gcloud storage cat {GCS}/manifest.json"))
staged   = manifest.get("staged_files")
items    = staged.items() if isinstance(staged, dict) else [(f.get("name"), f) for f in staged]

COLS = {}
for key, meta in items:
    if isinstance(meta, dict) and meta.get("column_order"):
        table = str(key).split("/")[-1].replace(".csv.gz", "").replace(".csv", "")
        COLS[table] = [c for c in meta["column_order"] if c not in PASS2]

for t in ORDER:
    assert t in COLS, f"{t}: no column_order in the manifest—never guess at this"
    proc = subprocess.Popen(["gcloud", "storage", "cat", f"{GCS}/{t}.csv.gz"],
                            stdout=subprocess.PIPE)
    with gzip.GzipFile(fileobj=proc.stdout, mode="rb") as gz:
        first = next(csv.reader(io.TextIOWrapper(gz, encoding="utf-8")))
    proc.stdout.close(); proc.wait()
    assert len(first) == len(COLS[t]), (
        f"{t}: file has {len(first)} fields, column list has {len(COLS[t])}. Refusing to load.")
    print(f"  {t:20s} {len(first):>3} fields  OK")

print("\npreflight passed—every file matches its column list")

### Load the eight relational tables

This is a client-side `COPY`: data streams from Cloud Storage, through this runtime, into AlloyDB.

`COPY` is PostgreSQL's bulk path, and it is dramatically faster than row-by-row `INSERT` because the
server parses a stream instead of planning and executing millions of individual statements. If you
take one habit home from this step, make it that one—reaching for `INSERT` in a loop is the single
most common reason a data load takes hours instead of minutes.

`appearances` is the big one at 832,193 rows and will dominate the time here. Expect a couple of
minutes for the whole step.

Note the explicit column list on every `COPY`. Positional loading works right up until someone adds a
column, at which point every field silently shifts by one.

In [ ]:
# Lab Task 1.4 — COPY the eight relational tables (~1.6M rows)

print("Loading eight tables. `appearances` is 832,193 rows and will take the longest.\n")

total0 = time.time()
for t in ORDER:
    existing = q(f"SELECT count(*) AS n FROM {t}")["n"][0]
    if existing:
        print(f"  {t:20s} already loaded—skipping{'':>12}{existing:>9,} rows")
        continue

    print(f"  {t:20s} loading...", end="", flush=True)
    t0 = time.time()
    copy_table(t, COLS[t], f"{GCS}/{t}.csv.gz")
    n = q(f"SELECT count(*) AS n FROM {t}")["n"][0]
    print(f"{'':>12}{n:>9,} rows  {time.time()-t0:>6.1f}s")

print(f"\npass 1 complete in {time.time()-total0:.1f}s")

### Load the profiles and their embeddings

Now the part that makes this lab possible.

Transfermarkt gives you numbers and categories: excellent for SQL, useless for searching by meaning.
So every player and club here also carries a **scouting profile**—roughly 250 words of narrative
prose describing how that player actually plays. Alongside it sits a **3,072-dimension embedding** of
that prose.

**All of it was generated once, offline, and staged.** Not to save you effort, but to make the lab
honest. Generating 14,235 grounded profiles costs real money and hours of wall clock, and generative
output drifts between runs. Pre-building means every student searches a byte-identical corpus—so
when the lab claims a particular query returns Neymar first, it does.

These land in staging tables and then `UPDATE` into place, because profiles are maintained separately
from the relational data and either can be regenerated without reloading the other.

In [ ]:
# Lab Task 1.4 — second pass: 14,235 profiles and their 3,072-dim embeddings

for t, key in [("players", "player_id"), ("clubs", "club_id")]:
    run(f"""CREATE TABLE IF NOT EXISTS _{t}_profiles (
                {key} INTEGER, profile_text TEXT, profile_embedding VECTOR(3072))""")
    if q(f"SELECT count(*) AS n FROM _{t}_profiles")["n"][0] == 0:
        t0 = time.time()
        copy_table(f"_{t}_profiles", [key, "profile_text", "profile_embedding"],
                   f"{GCS}/{t}_profiles.csv.gz")
        print(f"  staged {t} profiles in {time.time()-t0:.1f}s")

    run(f"""UPDATE {t} tgt
               SET profile_text      = src.profile_text,
                   profile_embedding = src.profile_embedding
              FROM _{t}_profiles src
             WHERE tgt.{key} = src.{key}""")

# Assert BOTH tables. A silent zero-row load is exactly the kind of failure that
# survives a happy-path check and then breaks Task 2 in front of a room.
n_players = q("SELECT count(*) AS n FROM players WHERE profile_embedding IS NOT NULL")["n"][0]
n_clubs   = q("SELECT count(*) AS n FROM clubs   WHERE profile_embedding IS NOT NULL")["n"][0]
assert n_players == 13439, f"expected 13,439 player profiles, got {n_players:,}"
assert n_clubs   == 796,   f"expected 796 club profiles, got {n_clubs:,}"
print(f"\n  {n_players:,} player profiles and {n_clubs:,} club profiles loaded and verified")

---

⏸ **Stop here.** You have finished the notebook's Task 1.4.

Return to the lab guide for **Task 1.5**, and take a look at what you just loaded.

## Task 1.5—Meet a vector embedding

### Look at what you just loaded

Before indexing any of it, spend thirty seconds on a single row. If "vector embedding" is a phrase
you have nodded along to without ever quite pinning down, this is the cell that fixes that.

In [ ]:
# Lab Task 1.5 — look at one embedding: a scouting report as 3,072 numbers

row = q("""
    SELECT player_name, main_position, profile_text,
           vector_dims(profile_embedding) AS dimensions,
           profile_embedding::text        AS full_vector
    FROM players
    WHERE player_name = 'Mauro Icardi' AND profile_embedding IS NOT NULL
    LIMIT 1
""")

r = row.iloc[0]
print(f"{r['player_name']}  ({r['main_position']})\n")
print("PROFILE TEXT—what a scout would write")
print("-" * 78)
print(textwrap.fill(r["profile_text"][:600].rsplit(" ", 1)[0] + " ...", 78))
print()
print(f"PROFILE EMBEDDING—the same meaning, as {r['dimensions']:,} numbers")
print("-" * 78)
head = r["full_vector"].strip("[]").split(",")[:8]
print("[" + ", ".join(f"{float(v):+.5f}" for v in head) + f", ... {r['dimensions'] - 8:,} more ]")

### So what is an embedding?

Say you watched *Casablanca* last night, and tonight someone says: "I'd like to watch a movie like *Casablanca*."

How would you build a system that answers that?

You might start by describing every movie as a list of numbers. Pick some dimensions and score each film on all of them:

| # | Dimension |
| --: | :---- |
| 1 | Year released |
| 2 | Rotten Tomatoes score |
| 3–5 | Three slots for the leading cast |
| 6 | Director |
| 7 | How much action, 0 to 1 |
| 8 | How much romance |
| 9 | How much comedy |
| 10 | How dark the ending is |

Now every movie is a list of ten numbers—a **vector**. And "like *Casablanca*" stops being a matter of taste and becomes arithmetic: find the films whose lists sit **closest** to *Casablanca*'s. Similar era. Overlapping cast. Comparable balance of romance to action. An ending that lands the same way.

Notice what you never had to write: a rule. Nobody coded "if it is a 1940s romance starring Bogart, recommend it." Similarity simply fell out of the numbers being near each other.

**That is an embedding.** The 3,072 numbers you just printed the first few elements of do exactly this for scouting reports instead of films.

With three differences that matter:

| Your movie vector | A real embedding |
| :---- | :---- |
| You chose the dimensions | The model learned them, from far more text than you will ever read |
| Every dimension means something you can name | **No dimension means anything you can name** |
| Ten dimensions | 3,072 of them |
| You would hand-score every film | The model reads the text and produces the numbers |

That second row is where people trip. You just built ten axes you could point at and explain. A real embedding has 3,072 axes and **not one of them is "finishing ability."** The model was never told what a striker is. It arranged the space so that documents used in similar ways end up in similar places, and the individual coordinates are a side effect of that arrangement, not a description of it.

So resist the urge to look up dimension 47 and ask what it represents. Only the *relative positions* carry meaning—which is exactly what the rest of this lab is built on.

Two more things worth holding onto:

- **3,072 is not a free choice here.** In Task 2 you will embed your own search text from inside SQL, and the model returns whatever width it natively produces. Stored vectors of any other width would fail the comparison outright.
- **It is not magic, and you will watch it fail.** Ask this corpus for *"someone who can unlock a parked bus"* and vector search confidently returns a **goalkeeper**. Hold on to that when Task 4 argues for combining methods rather than crowning one.

---

⏸ **Stop here.** You have finished the notebook's Task 1.5.

Return to the lab guide for **Task 1.6**, where you build the indexes—and find out why the order of these two steps is the most portable lesson in this task.

## Task 1.6—Build the indexes, now that the data has arrived

Order matters here, and it is the most portable lesson in Task 1.

**Build an index first, and every row you load afterwards has to be inserted into it one at a time.**
Load first, and the index is built once, in bulk, from data it can see in its entirety. That rule
holds for any bulk load into any indexed table you will ever own.

For ScaNN it is not merely faster, it is *required*. ScaNN works by clustering your vectors into
partitions and searching only the promising ones. On an empty table there is nothing to cluster, and
AlloyDB refuses outright:

```
FAILED_PRECONDITION: Cannot create ScaNN index with empty table "players"
```

The index definitions live in a staged file rather than being retyped here, so there is exactly one
authoritative copy of them. **Read them before you run them**—those ScaNN statements are the most
interesting DDL in this lab.

In [ ]:
# Lab Task 1.6 — read the index definitions before building them

indexes_sql = sh(f"gcloud storage cat {GCS}/indexes.sql")
assert indexes_sql.strip(), "could not read indexes.sql from Cloud Storage"
print(indexes_sql)

### What ScaNN actually is

**ScaNN** stands for **Scalable Nearest Neighbors**. It came out of Google Research, and it is the
same family of technique Google uses to search its own very large collections of vectors.

Here is the problem it solves. You have one query vector and 13,439 stored ones, and you want the
closest handful. The obvious approach is to compare your query against every single stored vector and
keep the best matches. That is a **brute-force scan**: perfectly accurate, and perfectly fine at
13,439 rows. At 13 million it is a disaster.

**So think about a bookstore instead.** A customer asks for something like the novel they just
finished. You do not read every book in the shop. You walk to the right section and look only at the
shelf in front of you.

ScaNN builds that store layout for you:

1. **At build time**, it groups the 13,439 vectors into 115 clusters, and computes one representative
   vector for each—the sign hanging over the aisle.
2. **At query time**, it compares your query against the 115 signs, not the 13,439 books. That part is
   cheap.
3. It then picks the closest few aisles and compares your query only against the vectors shelved
   there.

You just skipped the overwhelming majority of the comparisons.

**Which is exactly why it is called _approximate_ nearest neighbor.** A book can sit one aisle over
from where you looked. ScaNN can miss a genuinely close vector that happened to land in a cluster it
did not search. That is the trade: a small risk of missing something, for an enormous amount of work
avoided.

How much does it actually miss here? Nothing. On this corpus, the ten players ScaNN returns are the
same ten an exhaustive scan returns. **Approximate does not mean careless.**

### Now the statement makes sense

- **`USING scann (profile_embedding cosine)`**—index this column, measuring distance by the *angle*
  between vectors rather than their length. For text embeddings, direction is what carries meaning.
- **`num_leaves = 115`**—how many aisles to build. Roughly the square root of 13,439, and the square
  root is the balance point: too few aisles and each one is enormous, too many and reading all the
  signs costs as much as reading the books.
- **`quantizer = 'sq8'`**—store each dimension compressed to eight bits. A little precision traded for
  an index small enough to stay comfortably in memory, which is where you want it.

The six `btree` indexes in the same file are ordinary PostgreSQL, sitting on foreign key columns.
Worth knowing: **PostgreSQL does not index foreign keys automatically.** It indexes primary keys and
unique constraints; the child side of every relationship is yours to handle.

`maintenance_work_mem` gets raised first, because index builds use it as working space and the default
is far too small for this one.

In [ ]:
# Lab Task 1.6 — build ScaNN over the embeddings (AFTER the load, not before)

already = q("""
    SELECT count(*) AS n FROM pg_indexes
    WHERE schemaname = 'public' AND indexname LIKE '%scann%'
""")["n"][0]

if already >= 2:
    print("indexes already built—skipping (indexes.sql is not re-runnable)")
else:
    t0 = time.time()
    run("SET maintenance_work_mem = '2GB'")
    run(indexes_sql, show_notices=True)
    print(f"indexes built in {time.time()-t0:.1f}s")

display(q("""
    SELECT tablename, indexname
    FROM pg_indexes
    WHERE schemaname = 'public' AND tablename IN ('players','clubs')
    ORDER BY tablename, indexname
"""))

⚠️ **One thing that will look like a bug later.** If you add a narrow `WHERE` clause to a vector
search and ScaNN appears to stop being used, that is deliberate. Below roughly a thousand qualifying
rows the planner correctly decides that scanning the survivors directly is cheaper than consulting the
index. Nothing is broken—the optimizer is doing its job.

---

⏸ **Stop here.** You have finished the notebook's Task 1.6.

Return to the lab guide for **Task 1.7**, and verify what you built.

## Task 1.7—Verify, and record what you built

A quick census. These numbers are fixed: the corpus is a pinned snapshot, so everyone in the room sees
exactly these values, and so will you if you come back to this in two weeks.

In [ ]:
# Lab Task 1.7 — two censuses: what is searchable, and what is not

print("The searchable corpus—the only two tables carrying a text layer:\n")
display(q("""
    SELECT 'players' AS table_name, count(*) AS rows,
           count(profile_text) AS profiles, count(profile_embedding) AS embeddings
    FROM players
    UNION ALL
    SELECT 'clubs', count(*), count(profile_text), count(profile_embedding) FROM clubs
"""))

print("\nSupporting relational data—facts and figures, nothing to search by meaning:\n")
display(q("""
    SELECT 'competitions' AS table_name, count(*) AS rows FROM competitions
    UNION ALL SELECT 'games',             count(*) FROM games
    UNION ALL SELECT 'appearances',       count(*) FROM appearances
    UNION ALL SELECT 'game_events',       count(*) FROM game_events
    UNION ALL SELECT 'player_valuations', count(*) FROM player_valuations
    UNION ALL SELECT 'transfers',         count(*) FROM transfers
"""))

run("""CREATE TABLE IF NOT EXISTS provisioning_status (
         finished_at timestamptz PRIMARY KEY DEFAULT now(),
         players bigint, clubs bigint, appearances bigint)""")
run("""INSERT INTO provisioning_status (players, clubs, appearances)
       SELECT (SELECT count(*) FROM players WHERE profile_embedding IS NOT NULL),
              (SELECT count(*) FROM clubs   WHERE profile_embedding IS NOT NULL),
              (SELECT count(*) FROM appearances)""")

print("\nExpected: 13,439 players / 796 clubs / 832,193 appearances")
print("Task 1 complete. Your database is built, loaded, and indexed.")

---

⏸ **Stop here.** You have finished the notebook's Task 1.7.

**Task 1 is complete.** Your database is built, loaded, and indexed. Return to the lab guide for **Task 2**, where you go looking for two players and watch this corpus fail you twice.

---

# Tasks 2 through 5

From here the lab guide gives you each query and you add the cells. In this solution notebook they are already written.

Every code cell below opens with a comment naming its lab section.

## Task 2.1 — Ticket #4471, the transfer fee

Lab guide: **Task 2.1**. The search *works*, which is the whole problem.

In [ ]:
# Lab Task 2.1, step 1
# Ticket #4471 exactly as the fan typed it.
# It WORKS — one row, Neymar. That is what makes this bug survive triage.

q("""
    SELECT player_name, main_position, country_of_citizenship
    FROM players
    WHERE profile_text ILIKE '%€222,000,000%'
""")

In [ ]:
# Lab Task 2.1, step 2
# The same fact, five ways. Two of five match; three return nothing.
# The difference between success and failure is formatting, not meaning.

q("""
    SELECT '€222,000,000' AS as_typed, count(*) AS matches FROM players WHERE profile_text ILIKE '%€222,000,000%'
    UNION ALL SELECT '222,000,000',  count(*) FROM players WHERE profile_text ILIKE '%222,000,000%'
    UNION ALL SELECT '222000000',    count(*) FROM players WHERE profile_text ILIKE '%222000000%'
    UNION ALL SELECT '222 million',  count(*) FROM players WHERE profile_text ILIKE '%222 million%'
    UNION ALL SELECT '€222m',        count(*) FROM players WHERE profile_text ILIKE '%€222m%'
""")

## Task 2.2 — Ticket #4478, the striker

Lab guide: **Task 2.2**. No formatting trick rescues this one.

In [ ]:
# Lab Task 2.2, step 3
# Ticket #4478 exactly as the fan typed it. Zero rows.
# No scout ever wrote that sentence.

q("""
    SELECT player_name, main_position
    FROM players
    WHERE profile_text ILIKE '%a striker who gives centre-backs nightmares%'
""")

In [ ]:
# Lab Task 2.2, step 4
# How this corpus actually talks. Note: 'striker' 1,233 vs 'centre-back' 2,706.
# Those two numbers explain Task 3.4.

q("""
    SELECT 'striker' AS term,        count(*) AS profiles FROM players WHERE profile_text ILIKE '%striker%'
    UNION ALL SELECT 'centre-forward',  count(*) FROM players WHERE profile_text ILIKE '%centre-forward%'
    UNION ALL SELECT 'forward',         count(*) FROM players WHERE profile_text ILIKE '%forward%'
    UNION ALL SELECT 'attacker',        count(*) FROM players WHERE profile_text ILIKE '%attacker%'
    UNION ALL SELECT 'frontman',        count(*) FROM players WHERE profile_text ILIKE '%frontman%'
    UNION ALL SELECT 'centre-back',     count(*) FROM players WHERE profile_text ILIKE '%centre-back%'
""")

## Task 2.3 — The failure nobody files a ticket about

Lab guide: **Task 2.3**. Matching is not ranking.

In [ ]:
# Lab Task 2.3, step 5
# The failure nobody files a ticket about. Ten rows — but why THESE ten?
# There is no reason. LIKE returns a set, not a ranking.

q("""
    SELECT player_name, main_position
    FROM players
    WHERE profile_text ILIKE '%prolific goalscorer%'
    LIMIT 10
""")

---

## Task 3.1 — Build the BM25 index

Lab guide: **Task 3.1**. One statement, about three seconds, and it narrates itself.

In [ ]:
# Lab Task 3.1, step 1
# Build the BM25 index. Requires alloydbsuperuser (granted at provisioning).
# Read the NOTICE output: k1=1.20, b=0.75, avg_length=161.68.

run("""
    CREATE INDEX players_profile_text_bm25_idx
    ON players USING bm25 (profile_text)
    WITH (text_config = 'english')
""", show_notices=True)

## Task 3.2 — Ticket #4471, solved

Lab guide: **Task 3.2**. This is where the `QUERY` variable convention starts.

In [ ]:
# Lab Task 3.2, step 3
# Ticket #4471 through BM25. Neymar first at about -12.2; second place about -4.6.
# NOTE the QUERY variable — every search from here on is written this way.

QUERY = "€222,000,000"

q(f"""
    SELECT player_name, main_position,
           profile_text <@> '{QUERY}' AS bm25_score
    FROM players
    WHERE profile_text <@> '{QUERY}' < 0
    ORDER BY profile_text <@> '{QUERY}'
    LIMIT 5
""")

In [ ]:
# Lab Task 3.2, step 4
# How many rows actually matched? About 4,036 — not one.
# BM25 did not FIND the document. It ranked a huge pile so well that yours came first.

q(f"""
    SELECT count(*) AS rows_matched
    FROM players
    WHERE profile_text <@> '{QUERY}' < 0
""")

## Task 3.3 — Prove the index is doing the work

Lab guide: **Task 3.3**. `<@>` has two overloads and only one uses the index.

In [ ]:
# Lab Task 3.3, step 5
# Prove the index is doing the work. Look for:
#   Index Scan using players_profile_text_bm25_idx
# A Seq Scan means <@> resolved to the non-indexed overload.

explain(f"""
    SELECT player_name, profile_text <@> '{QUERY}' AS bm25_score
    FROM players
    WHERE profile_text <@> '{QUERY}' < 0
    ORDER BY profile_text <@> '{QUERY}'
    LIMIT 5
""")

## Task 3.4 — Ticket #4478, still broken

Lab guide: **Task 3.4**. BM25 fails here predictably, which is what makes it useful.

In [ ]:
# Lab Task 3.4, step 6
# Ticket #4478 through BM25. The top result is a DEFENDER.
# 'centre-back' is the rarer term, so BM25 followed it to the wrong position group.

QUERY = "a striker who gives centre-backs nightmares"

q(f"""
    SELECT player_name, main_position,
           profile_text <@> '{QUERY}' AS bm25_score
    FROM players
    WHERE profile_text <@> '{QUERY}' < 0
    ORDER BY profile_text <@> '{QUERY}'
    LIMIT 5
""")

In [ ]:
# Lab Task 3.4, step 7
# About 10,700 rows matched — the better part of the corpus.

q(f"""
    SELECT count(*) AS rows_matched
    FROM players
    WHERE profile_text <@> '{QUERY}' < 0
""")

---

## Task 4.1 — Introducing the other half

Lab guide: **Task 4.1**. Vector search, and the query it cannot answer.

In [ ]:
# Lab Task 4.1, step 1
# The same striker query through the vectors instead of the text.
# Five attackers, mostly Centre-Forward. BM25 gave a defender.

QUERY = "a striker who gives centre-backs nightmares"

q(f"""
    SELECT player_name, main_position, detailed_position,
           profile_embedding <=> ai.embedding('gemini-embedding-001',
               '{QUERY}')::vector AS distance
    FROM players
    WHERE profile_embedding IS NOT NULL
    ORDER BY profile_embedding <=> ai.embedding('gemini-embedding-001',
               '{QUERY}')::vector
    LIMIT 5
""")

In [ ]:
# Lab Task 4.1, step 2
# Do those profiles even contain the fan's words? Mostly not.
# The ranking never depended on matching a word.

q(f"""
    WITH top5 AS (
        SELECT player_id,
               profile_embedding <=> ai.embedding('gemini-embedding-001',
                   '{QUERY}')::vector AS distance
        FROM players
        WHERE profile_embedding IS NOT NULL
        ORDER BY 2
        LIMIT 5
    )
    SELECT p.player_name, p.main_position,
           p.profile_text ILIKE '%striker%'     AS says_striker,
           p.profile_text ILIKE '%centre-back%' AS says_centre_back,
           p.profile_text ILIKE '%nightmare%'   AS says_nightmare
    FROM top5 t JOIN players p USING (player_id)
    ORDER BY t.distance
""")

In [ ]:
# Lab Task 4.1, step 3
# Vector search's worst case: hand it the transfer fee.
# Five confident players, and Neymar is almost certainly not among them.

QUERY = "€222,000,000"

q(f"""
    SELECT player_name, main_position,
           profile_embedding <=> ai.embedding('gemini-embedding-001',
               '{QUERY}')::vector AS distance
    FROM players
    WHERE profile_embedding IS NOT NULL
    ORDER BY profile_embedding <=> ai.embedding('gemini-embedding-001',
               '{QUERY}')::vector
    LIMIT 5
""")

## Task 4.2 — The built-in that cannot help you

Lab guide: **Task 4.2**. Ask the database what a function looks like before you call it.

In [ ]:
# Lab Task 4.2, step 4
# Introspect ai.hybrid_search() BEFORE calling it.
# It accepts only ts_rank and RUM's <=> — neither of which is BM25.

sigs = q("""
    SELECT pg_get_function_arguments(p.oid) AS arguments
    FROM pg_proc p
    JOIN pg_namespace n ON n.oid = p.pronamespace
    WHERE n.nspname = 'ai' AND p.proname = 'hybrid_search'
""")

for a in sigs["arguments"]:
    print(a.replace(", ", ",\n    "), "\n")

In [ ]:
# Lab Task 4.2, step 5
# Is RUM even available? No. Which is why this lab teaches BM25.

q("""
    SELECT name, default_version, installed_version
    FROM pg_available_extensions
    WHERE name IN ('rum', 'pg_textsearch')
""")

## Task 4.3 — Build the fusion yourself

Lab guide: **Task 4.3**. Reciprocal Rank Fusion, `k = 60`, the same algorithm the built-in uses.

In [ ]:
# Lab Task 4.3, step 6
# The vector half, ranked.
# Note the shape: filter/sort/LIMIT inside, number the survivors outside.

QUERY = "a striker who gives centre-backs nightmares"

q(f"""
    WITH hits AS (
        SELECT player_id, player_name,
               profile_embedding <=> ai.embedding(
                   'gemini-embedding-001',
                   '{QUERY}')::vector AS distance
        FROM players
        WHERE profile_embedding IS NOT NULL
        ORDER BY distance
        LIMIT 10
    )
    SELECT player_id, player_name, distance,
           ROW_NUMBER() OVER (ORDER BY distance) AS rank
    FROM hits
    ORDER BY rank
""")

In [ ]:
# Lab Task 4.3, step 7
# Corpus census for the fee query:
#   4,036 real matches, 9,403 rows scoring exactly 0, 13,439 total.
# The zeros are queued up behind your real matches, ready to fill any spare slot.

q("""
    SELECT
        count(*) FILTER (WHERE profile_text <@> '€222,000,000' < 0) AS real_matches,
        count(*) FILTER (WHERE profile_text <@> '€222,000,000' = 0) AS zero_score_rows,
        count(*) AS corpus
    FROM players
""")

In [ ]:
# Lab Task 4.3, step 8
# The text half, done right. WHERE ... < 0 is the most important line in Task 4.

q(f"""
    WITH hits AS (
        SELECT player_id, player_name,
               profile_text <@> '{QUERY}' AS score
        FROM players
        WHERE profile_text <@> '{QUERY}' < 0
        ORDER BY score
        LIMIT 10
    )
    SELECT player_id, player_name, score,
           ROW_NUMBER() OVER (ORDER BY score) AS rank
    FROM hits
    ORDER BY rank
""")

In [ ]:
# Lab Task 4.3, step 8
# DELIBERATELY WRONG — kept because it is the lesson.
# The search string appears THREE times. Change two and miss the third and you get
# a perfect score column beside a rank column answering the previous question.
# Nothing errors.

q("""
    SELECT player_id, player_name,
           profile_text <@> '€222,000,000' AS score,
           ROW_NUMBER() OVER (ORDER BY profile_text <@> '€222,000,000') AS rank
    FROM players
    ORDER BY profile_text <@> '€222,000,000'
    LIMIT 10
""")

In [ ]:
# Lab Task 4.3, step 8
# The fix, and the habit: one QUERY variable feeding every copy.
# Try setting QUERY to the fee and re-running — Neymar arrives at rank 1.

QUERY = "a striker who gives centre-backs nightmares"

q(f"""
    WITH hits AS (
        SELECT player_id, player_name,
               profile_text <@> '{QUERY}' AS score
        FROM players
        WHERE profile_text <@> '{QUERY}' < 0
        ORDER BY score
        LIMIT 10
    )
    SELECT player_id, player_name, score,
           ROW_NUMBER() OVER (ORDER BY score) AS rank
    FROM hits
    ORDER BY rank
""")

In [ ]:
# Lab Task 4.3, step 9
# THE FUSION. Retrieve 50 per list, fuse 100 candidates, cut to 10.
# Depth matters: at 10 the two engines share NO players at all.
# Expect one or two players with a number in BOTH rank columns, at the very top.
# Whoever they are, they were nowhere near first in either list. Any player found
# by both outranks every player found by only one: the best a single-list hit can
# score is 1/61 = 0.0164, while two fiftieth places still make 0.0182.

QUERY = "a striker who gives centre-backs nightmares"

q(f"""
    WITH vector_hits AS (
        SELECT player_id,
               profile_embedding <=> ai.embedding(
                   'gemini-embedding-001',
                   '{QUERY}')::vector AS distance
        FROM players
        WHERE profile_embedding IS NOT NULL
        ORDER BY distance
        LIMIT 50
    ),
    vector_search AS (
        SELECT player_id, ROW_NUMBER() OVER (ORDER BY distance) AS rank
        FROM vector_hits
    ),
    text_hits AS (
        SELECT player_id,
               profile_text <@> '{QUERY}' AS score
        FROM players
        WHERE profile_text <@> '{QUERY}' < 0
        ORDER BY score
        LIMIT 50
    ),
    text_search AS (
        SELECT player_id, ROW_NUMBER() OVER (ORDER BY score) AS rank
        FROM text_hits
    )
    SELECT p.player_name, p.main_position,
           v.rank AS vector_rank,
           t.rank AS text_rank,
           COALESCE(1.0 / (60 + v.rank), 0.0)
         + COALESCE(1.0 / (60 + t.rank), 0.0) AS rrf_score
    FROM vector_search v
    FULL OUTER JOIN text_search t ON v.player_id = t.player_id
    JOIN players p ON p.player_id = COALESCE(v.player_id, t.player_id)
    ORDER BY rrf_score DESC
    LIMIT 10
""")

## Task 4.4 — Does it actually fix both tickets?

Lab guide: **Task 4.4**.

In [ ]:
# Lab Task 4.4, step 11
# The SAME fused query with one line changed. That is the payoff for the variable.
# Neymar arrives at the TOP, tied with vector search's best guess - lifted level on
# the strength of one list. RRF only knows positions, so it cannot prefer one
# first place over another. A player found by BOTH engines outranks him, which
# is the rule working, not a fault. Task 5 sorts the order out.

QUERY = "€222,000,000"

q(f"""
    WITH vector_hits AS (
        SELECT player_id,
               profile_embedding <=> ai.embedding(
                   'gemini-embedding-001',
                   '{QUERY}')::vector AS distance
        FROM players
        WHERE profile_embedding IS NOT NULL
        ORDER BY distance
        LIMIT 50
    ),
    vector_search AS (
        SELECT player_id, ROW_NUMBER() OVER (ORDER BY distance) AS rank
        FROM vector_hits
    ),
    text_hits AS (
        SELECT player_id,
               profile_text <@> '{QUERY}' AS score
        FROM players
        WHERE profile_text <@> '{QUERY}' < 0
        ORDER BY score
        LIMIT 50
    ),
    text_search AS (
        SELECT player_id, ROW_NUMBER() OVER (ORDER BY score) AS rank
        FROM text_hits
    )
    SELECT p.player_name, p.main_position,
           v.rank AS vector_rank,
           t.rank AS text_rank,
           COALESCE(1.0 / (60 + v.rank), 0.0)
         + COALESCE(1.0 / (60 + t.rank), 0.0) AS rrf_score
    FROM vector_search v
    FULL OUTER JOIN text_search t ON v.player_id = t.player_id
    JOIN players p ON p.player_id = COALESCE(v.player_id, t.player_id)
    ORDER BY rrf_score DESC
    LIMIT 10
""")

In [ ]:
# Lab Task 4.4, step 12
# Confirm the BM25 index is still used once the query is wrapped in a CTE.

explain(f"""
    WITH hits AS (
        SELECT player_id,
               profile_text <@> '{QUERY}' AS score
        FROM players
        WHERE profile_text <@> '{QUERY}' < 0
        ORDER BY score
        LIMIT 10
    )
    SELECT player_id, ROW_NUMBER() OVER (ORDER BY score) AS rank
    FROM hits
""")

---

## Task 5.1 — Ask the database what it can call

Lab guide: **Task 5.1**.

In [ ]:
# Lab Task 5.1, step 1
# Which reranking models can this database reach?
# Five, pre-registered. Note model_request_url — these are Discovery Engine,
# not Vertex, which is why the service agent needs roles/discoveryengine.viewer.

q("""
    SELECT model_id, model_type, model_request_url
    FROM google_ml.model_info_view
    WHERE model_type = 'reranking'
    ORDER BY model_id
""")

## Task 5.2 — Read the signature before you call it

Lab guide: **Task 5.2**. The return type is the trap.

In [ ]:
# Lab Task 5.2, step 2
# ai.rank() signatures. Four overloads; the reranker is the one taking documents text[].
# It returns TABLE(index integer, score real) — POSITIONS, not rows. Join back.
# index is 1-based. Score is higher-is-better (the third convention in this lab).

sigs = q("""
    SELECT pg_get_function_arguments(p.oid) AS arguments,
           pg_get_function_result(p.oid)    AS returns
    FROM pg_proc p
    JOIN pg_namespace n ON n.oid = p.pronamespace
    WHERE n.nspname = 'ai' AND p.proname = 'rank'
""")

for _, r in sigs.iterrows():
    print(r["arguments"].replace(", ", ",\n    "), "\n  ->", r["returns"], "\n")

## Task 5.3 — Rerank the shortlist

Lab guide: **Task 5.3**.

In [ ]:
# Lab Task 5.3, step 3
# THE FULL PIPELINE: fuse 100 candidates, shortlist 10, rerank by reading them.
# Read the result from the BOTTOM: Souleymane Diawara (a DEFENDER, and BM25's #1)
# lands last at about 0.12, well clear of everyone else.
# Then the top: the winner came from near the BOTTOM of the fused list.

MODEL = "semantic-ranker-default-004"
QUERY = "a striker who gives centre-backs nightmares"

q(f"""
    WITH vector_hits AS (
        SELECT player_id, profile_embedding <=> ai.embedding(
                   'gemini-embedding-001', '{QUERY}')::vector AS distance
        FROM players WHERE profile_embedding IS NOT NULL
        ORDER BY distance LIMIT 50
    ),
    vector_search AS (
        SELECT player_id, ROW_NUMBER() OVER (ORDER BY distance) AS rank
        FROM vector_hits
    ),
    text_hits AS (
        SELECT player_id, profile_text <@> '{QUERY}' AS score
        FROM players WHERE profile_text <@> '{QUERY}' < 0
        ORDER BY score LIMIT 50
    ),
    text_search AS (
        SELECT player_id, ROW_NUMBER() OVER (ORDER BY score) AS rank
        FROM text_hits
    ),
    fused AS (
        SELECT COALESCE(v.player_id, t.player_id) AS player_id,
               COALESCE(1.0 / (60 + v.rank), 0.0)
             + COALESCE(1.0 / (60 + t.rank), 0.0) AS rrf_score
        FROM vector_search v
        FULL OUTER JOIN text_search t ON v.player_id = t.player_id
        ORDER BY rrf_score DESC
        LIMIT 10
    ),
    docs AS (
        SELECT array_agg(left(p.profile_text, 1200) ORDER BY f.rrf_score DESC) AS d,
               array_agg(p.player_name             ORDER BY f.rrf_score DESC) AS n,
               array_agg(p.main_position           ORDER BY f.rrf_score DESC) AS pos
        FROM fused f JOIN players p USING (player_id)
    )
    SELECT r.index            AS came_from_fused_rank,
           r.score            AS rerank_score,
           (docs.n)[r.index]  AS player_name,
           (docs.pos)[r.index] AS main_position
    FROM docs, ai.rank('{MODEL}', '{QUERY}', docs.d, 10) AS r
    ORDER BY r.score DESC
""")

## Task 5.4 — Ticket #4471, finished

Lab guide: **Task 5.4, step 6**.

In [ ]:
# Lab Task 5.4, step 6
# Neymar FIRST at about 0.35, second place under 0.09 — a four-fold margin.
# Compare the score SHAPES between this and the striker query above:
#   striker  ~0.50 top, rest 0.22-0.45  -> many good answers, model says so
#   fee      ~0.35 top, rest 0.02-0.08  -> one right answer, found
# That distribution is a confidence signal you did not have to build.

MODEL = "semantic-ranker-default-004"
QUERY = "€222,000,000"

q(f"""
    WITH vector_hits AS (
        SELECT player_id, profile_embedding <=> ai.embedding(
                   'gemini-embedding-001', '{QUERY}')::vector AS distance
        FROM players WHERE profile_embedding IS NOT NULL
        ORDER BY distance LIMIT 50
    ),
    vector_search AS (
        SELECT player_id, ROW_NUMBER() OVER (ORDER BY distance) AS rank
        FROM vector_hits
    ),
    text_hits AS (
        SELECT player_id, profile_text <@> '{QUERY}' AS score
        FROM players WHERE profile_text <@> '{QUERY}' < 0
        ORDER BY score LIMIT 50
    ),
    text_search AS (
        SELECT player_id, ROW_NUMBER() OVER (ORDER BY score) AS rank
        FROM text_hits
    ),
    fused AS (
        SELECT COALESCE(v.player_id, t.player_id) AS player_id,
               COALESCE(1.0 / (60 + v.rank), 0.0)
             + COALESCE(1.0 / (60 + t.rank), 0.0) AS rrf_score
        FROM vector_search v
        FULL OUTER JOIN text_search t ON v.player_id = t.player_id
        ORDER BY rrf_score DESC
        LIMIT 10
    ),
    docs AS (
        SELECT array_agg(left(p.profile_text, 1200) ORDER BY f.rrf_score DESC) AS d,
               array_agg(p.player_name             ORDER BY f.rrf_score DESC) AS n,
               array_agg(p.main_position           ORDER BY f.rrf_score DESC) AS pos
        FROM fused f JOIN players p USING (player_id)
    )
    SELECT r.index            AS came_from_fused_rank,
           r.score            AS rerank_score,
           (docs.n)[r.index]  AS player_name,
           (docs.pos)[r.index] AS main_position
    FROM docs, ai.rank('{MODEL}', '{QUERY}', docs.d, 10) AS r
    ORDER BY r.score DESC
""")

---

## Where to go next

The **Optional** section at the end of the lab guide has six open questions with no answers written down—rescuing the *parked bus* query, searching the untouched `clubs` corpus, fusing in a third signal, mapping the retrieval-depth curve, comparing the five rerankers, and breaking RRF by changing `k`.

This repository is yours to keep. The `terraform/` folder built your cluster and the other notebooks in this folder generated the corpus.